# 05. 공식 데이터만으로 하는 모델 비교

**이 노트북은 공식 문제 데이터 4개 파일만 씁니다.** `labeled_data.csv` 같은 외부데이터는 한 줄도 쓰지 않습니다.
여기서 나온 숫자가 보고서 본문(A층)의 숫자가 됩니다.

### 진행 방식
각 단계마다

1. **읽기** — 왜 하는지, 알아야 할 사실
2. **질문** — 실행 *전에* 예상 답을 아래 빈칸에 적어보기
3. **실행**
4. **확인** — 예상이 맞았는지 보기

질문은 `> 질문` 으로 표시했습니다. 답은 바로 아래 `내 답:` 칸에 적으세요.

---
## 0. 데이터 불러오기와 쌍 구조

### 알아야 할 사실 세 가지

**① 값이 표준화되어 있습니다.** 출제측이 각 변수를 평균 0, 표준편차 1 근처로 바꿔놨습니다.
`Injection_Time = -0.03` 은 "0.03초"가 아니라 "평균보다 0.03 표준편차 짧다"는 뜻입니다.

**② `Unnamed: 0` 은 행 순서이고, 시간 순서를 보존합니다.** 이름을 `order` 로 바꿔 씁니다.
라벨 파일은 0~2392 연속, 비라벨 파일은 원본 비라벨 파일의 행 번호입니다.

**③ 바로 옆 행과 24개 값이 같으면 한 쇼트의 두 부품(좌/우)입니다.**
사출 한 번에 부품 두 개가 나오고, 기계는 쇼트 단위로 값을 기록하니 두 부품이 같은 값을 받습니다.
이 둘을 같은 `pair` 번호로 묶습니다.

> **질문 0-1.** 값이 표준화되어 있으면, 할 수 없게 되는 분석은 무엇일까요? (힌트: 단원에서 배운 물리 관계식)
>
> 내 답:

> **질문 0-2.** `pair` 번호를 굳이 만드는 이유는? 안 만들고 그냥 행 단위로 train/test를 나누면 무슨 일이 생길까요?
>
> 내 답:

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.load import RAW

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

FILES = {"CN7": ("moldset_labeled_cn7.csv", "moldset_unlabeled_cn7.csv"),
         "RG3": ("moldset_labeled_rg3.csv", "moldset_unlabeled_rg3.csv")}

def load_official(prod):
    lab = pd.read_csv(RAW / FILES[prod][0]).rename(columns={"Unnamed: 0": "order"})
    unl = pd.read_csv(RAW / FILES[prod][1]).rename(columns={"Unnamed: 0": "order"})
    lab = lab.rename(columns={"PassOrFail": "y"})
    return lab.sort_values("order").reset_index(drop=True), unl.sort_values("order").reset_index(drop=True)

VARS = [c for c in pd.read_csv(RAW / FILES["CN7"][0], nrows=1).columns
        if c not in ("Unnamed: 0", "PassOrFail")]
print(f"공정변수 {len(VARS)}개:", VARS)

In [ ]:
def add_pair(df):
    """바로 옆 행과 24개 값이 완전히 같으면 같은 쇼트(pair)로 묶는다."""
    k = pd.util.hash_pandas_object(df[VARS], index=False).values
    pid = np.empty(len(df), dtype=int)
    p = i = 0
    while i < len(df):
        pid[i] = p
        if i + 1 < len(df) and k[i] == k[i + 1]:
            pid[i + 1] = p
            i += 2
        else:
            i += 1
        p += 1
    df = df.copy()
    df["pair"] = pid
    df["key"] = k            # 24개 값이 같은 행은 같은 key
    return df

DATA = {}
rows = []
for prod in ["CN7", "RG3"]:
    lab, unl = load_official(prod)
    lab, unl = add_pair(lab), add_pair(unl)
    DATA[prod] = (lab, unl)

    size = lab.groupby("pair").size()
    ybad = lab.groupby("pair")["y"].sum()
    rows.append({"제품": prod,
                 "라벨 행": len(lab), "불량 행": int(lab["y"].sum()),
                 "쌍(2행)": int((size == 2).sum()), "단독(1행)": int((size == 1).sum()),
                 "양쪽 다 불량 쌍": int(((size == 2) & (ybad == 2)).sum()),
                 "한쪽만 불량 쌍(쌍둥이)": int(((size == 2) & (ybad == 1)).sum()),
                 "비라벨 행": len(unl),
                 "비라벨 쌍": int((unl.groupby('pair').size() == 2).sum())})
pd.DataFrame(rows).set_index("제품").T

### 확인
- 라벨 파일은 행 대부분이 쌍으로 묶이는지
- 비라벨 파일은 왜 쌍 비율이 훨씬 낮은지 (아직 원인 모름 — 기억해두기)
- "한쪽만 불량 쌍"이 RG3에서 몇 개인지

아래는 변수별로 값이 몇 종류인지입니다. 거의 안 변하는 변수는 모델에 정보를 못 줍니다.

In [ ]:
nu = pd.DataFrame({prod: DATA[prod][0][VARS].nunique() for prod in ["CN7", "RG3"]})
nu["비고"] = np.where(nu.min(axis=1) <= 1, "상수 (정보 없음)",
              np.where(nu.min(axis=1) <= 3, "값 2~3개뿐", ""))
nu.sort_values("CN7")

---
## 1. 이론 상한 F1 — 어떤 모델도 넘을 수 없는 선

### 개념
모델은 **같은 입력에는 같은 답**을 냅니다. 24개 값이 완전히 같은 두 행이 있으면, 모델은 둘 다 불량이라 하거나 둘 다 양품이라 할 수밖에 없습니다.

쌍둥이(같은 입력, 하나는 불량·하나는 양품) 한 쌍을 생각해보면

| 모델의 판정 | TP | FP | FN |
|---|---|---|---|
| 둘 다 "불량" | 1 | 1 | 0 |
| 둘 다 "양품" | 0 | 0 | 1 |

어느 쪽을 골라도 하나는 틀립니다.

그래서 **같은 입력을 가진 행들을 한 묶음**으로 보고, 어떤 묶음을 "불량"으로 판정해야 F1이 최대가 되는지 계산하면
그게 **정답을 다 아는 모델이라도 넘을 수 없는 F1**입니다.

> **질문 1-1.** 위 표에서 F1만 생각하면 쌍둥이 쌍을 "둘 다 불량"으로 판정하는 게 나을까요, "둘 다 양품"이 나을까요? 이유는?
>
> 내 답:

> **질문 1-2.** RG3 불량 25건이 전부 쌍둥이라면 이론 상한 F1은 대략 얼마일지 손으로 계산해보세요. (TP, FP, FN을 세어서)
>
> 내 답:

> **질문 1-3.** 가이드북은 RG3에서 F1 0.99를 보고했습니다. 이 노트북에서 나올 상한보다 높다면, 무엇을 뜻할까요?
>
> 내 답:

In [ ]:
def f1_ceiling(lab):
    """같은 입력(key)끼리는 같은 판정을 받는다는 제약에서 달성 가능한 최대 F1."""
    g = lab.groupby("key")["y"].agg(p="sum", t="size")
    g["n"] = g["t"] - g["p"]
    g = g[g["p"] > 0].copy()                 # 불량이 하나라도 있는 묶음만 후보
    P = lab["y"].sum()
    g["ratio"] = g["p"] / g["t"]
    g = g.sort_values(["ratio", "p"], ascending=False)

    best = dict(F1=0.0)
    tp = fp = 0
    for _, r in g.iterrows():               # 불량 비율 높은 묶음부터 하나씩 "불량"으로 판정
        tp += r["p"]; fp += r["n"]
        f1 = 2 * tp / (2 * tp + fp + (P - tp))
        if f1 > best["F1"]:
            best = dict(F1=f1, TP=int(tp), FP=int(fp), FN=int(P - tp),
                        정밀도=tp / (tp + fp), 재현율=tp / P)
    return best

CEIL = {}
for prod in ["CN7", "RG3"]:
    CEIL[prod] = f1_ceiling(DATA[prod][0])
pd.DataFrame(CEIL).T.round(3)

### 확인
질문 1-2에서 손으로 계산한 값과 같은지 보세요.

이 숫자가 이후 모든 모델 성능의 **분모**가 됩니다. 보고서에는 "F1 0.4"가 아니라 "상한 0.667 대비 60%"처럼 씁니다.

---
## 2. 평가 틀 정하기

모델을 만들기 **전에** 어떻게 잴지를 고정합니다. 나중에 정하면 결과를 보고 유리한 방식을 고르게 되거든요.

### 분할 방식 두 가지

**A. 쌍 단위 층화 분할** — 같은 쌍은 반드시 같은 fold에. 불량은 fold마다 고르게.
좌우 부품 누설은 막지만, 같은 시간대의 쇼트들이 train과 test에 섞입니다.

**B. 시간 블록 분할** — 행 순서대로 연속된 5구간으로 자름 (쌍은 안 깨지게).
"과거로 학습해서 처음 보는 시간대를 예측"하는 상황. 현장과 가장 비슷합니다.

> **질문 2-1.** CN7 불량 17건은 행 번호 47, 80, 95~120에 있습니다(전체 0~1210). 시간 블록 5개로 나누면 불량이 어떻게 흩어질까요? 그때 랜덤포레스트 같은 지도학습 모델에는 무슨 일이 생길까요?
>
> 내 답:

> **질문 2-2.** 비라벨 데이터로만 "정상"을 배우는 이상탐지 모델은 B 방식에서 왜 영향을 덜 받을까요?
>
> 내 답:

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

K = 5

def make_folds(lab, kind, seed=42):
    fold = np.full(len(lab), -1)
    if kind == "A_쌍층화":
        cv = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=seed)
        for f, (_, te) in enumerate(cv.split(lab, lab["y"], lab["pair"])):
            fold[te] = f
    elif kind == "B_시간블록":
        pairs = lab["pair"].unique()                 # 이미 시간 순
        for f, chunk in enumerate(np.array_split(pairs, K)):
            fold[lab["pair"].isin(chunk).values] = f
    return fold

for prod in ["CN7", "RG3"]:
    lab = DATA[prod][0]
    print(f"\n── {prod} ──")
    for kind in ["A_쌍층화", "B_시간블록"]:
        lab[kind] = make_folds(lab, kind)
        t = lab.groupby(kind).agg(행=("y", "size"), 불량=("y", "sum"),
                                 행번호=("order", lambda s: f"{s.min()}~{s.max()}"))
        # 쌍이 fold를 넘어가지 않는지 검사
        broken = (lab.groupby("pair")[kind].nunique() > 1).sum()
        print(f"[{kind}]  쌍이 갈라진 경우: {broken}개")
        print(t.T.to_string())

### 지표 세 가지

| 지표 | 무엇 | 임계값 |
|---|---|---|
| **AP** (평균 정밀도) | 점수 순서가 불량을 얼마나 위로 올리는가 | 필요 없음 |
| **상위 k% 검출률** | 상위 1·2·5%만 검사하면 불량의 몇 %가 잡히나 | k로 고정 (현장 운영 방식) |
| **F1** | 최적 임계값에서의 F1 → 이론 상한과 비교 | 결과를 보고 고름 → **낙관적 참고치** |

무작위 점수의 AP는 불량 비율과 같습니다 (CN7 0.014, RG3 0.021). 그래서 AP는 "무작위 대비 몇 배"로도 같이 봅니다.

> **질문 2-3.** F1의 최적 임계값을 test 결과를 보고 고르면 왜 "낙관적"일까요?
>
> 내 답:

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

def evaluate(lab, score, name, prod):
    """점수(높을수록 불량 의심)를 받아 지표를 계산한다. score는 lab과 같은 순서."""
    y = lab["y"].values
    s = np.asarray(score, dtype=float)
    ok = ~np.isnan(s)
    y, s = y[ok], s[ok]

    ap = average_precision_score(y, s)
    base = y.mean()

    p, r, _ = precision_recall_curve(y, s)
    f1 = np.nanmax(2 * p * r / np.where(p + r == 0, np.nan, p + r))

    out = {"모델": name, "제품": prod, "채점 행": int(ok.sum()),
           "AP": ap, "무작위 대비": ap / base,
           "F1(최적, 참고)": f1, "상한 대비": f1 / CEIL[prod]["F1"]}
    order = np.argsort(-s)
    for k in [1, 2, 5]:
        top = order[: max(1, int(len(s) * k / 100))]
        out[f"상위{k}% 검출률"] = y[top].sum() / y.sum()
    return out

# 기준점: 무작위 점수
RESULTS = []
rng = np.random.RandomState(0)
for prod in ["CN7", "RG3"]:
    lab = DATA[prod][0]
    RESULTS.append(evaluate(lab, rng.rand(len(lab)), "무작위", prod))
pd.DataFrame(RESULTS).round(3)

### 확인
무작위 점수의 AP가 불량 비율 근처, 상위 k% 검출률이 대략 k% 근처인지 보세요.
이게 **0점 기준선**입니다. 앞으로 모든 모델은 이 줄과 비교합니다.

---
## 여기까지 하고 멈춤

질문 0-1 ~ 2-3에 답을 적고, 실행 결과(표 3개)를 공유해주세요.
다음 단계(3. 랜덤포레스트)로 넘어갑니다.